# CS231 - Thành viên 2: Mạng nơ-ron Tích chập Sâu (CNN) & Giải thích AI (Grad-CAM)


In [ ]:
import sys, os
sys.path.append('..')

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.member2_deeplearning.dataset_aug import WasteDataset, get_transforms
from src.member2_deeplearning.train_cnn import train_cnn_model
from src.base_predictor import CLASS_NAMES

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())


### 1. Kiểm tra Pipeline Tăng cường Dữ liệu (Data Augmentation)


In [ ]:
train_dir = '../data/standard_split/train'
dataset = WasteDataset(train_dir, class_names=CLASS_NAMES, transform=get_transforms(is_train=True))

if len(dataset) > 0:
    tensor, label = dataset[0]
    print(f"Tensor shape: {tensor.shape}, Label: {CLASS_NAMES[label]}")


### 2. Huấn luyện Mô hình CNN


In [ ]:
# Huấn luyện mạng CNN và lưu trọng số tốt nhất vào weights/mobilenetv4_best.pth
train_cnn_model(config_path='../configs/config.yaml', model_name='light_cnn', epochs=15, batch_size=32)


### 3. Trực quan hóa Giải thích bằng Grad-CAM


In [ ]:
from src.member2_deeplearning.predictor_dl import DeepLearningPredictor

predictor = DeepLearningPredictor(weight_path='../weights/mobilenetv4_best.pth')

import glob
test_samples = glob.glob('../data/standard_split/test/plastic/*.*')
if test_samples:
    bgr = cv2.imread(test_samples[0])
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_224 = cv2.resize(rgb, (224, 224))
    
    result = predictor.predict(rgb_224)
    print("CNN Predicts:", result['class_name'], "| Conf:", result['confidence'])
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(rgb_224); ax[0].set_title("Ảnh đầu vào (Plastic Bottle)"); ax[0].axis('off')
    ax[1].imshow(result['visual_explanation']); ax[1].set_title("Vùng chú ý Grad-CAM"); ax[1].axis('off')
    plt.tight_layout()
    plt.show()
